## Avaliações preliminares para escolhas técnicas:

In [13]:
import struct
path='../../embeddings/bio_embedding_extrinsic'
with open(path, 'rb') as f:
    header = f.readline().decode('utf-8')      # "2324849 200\n"
    vocab_size, dim = map(int, header.split())
    word = f.read(4).decode('utf-8')            # "the "
    vec_bytes = f.read(dim * 4)
    vec = struct.unpack(f'<{dim}f', vec_bytes)   # 200 floats, little-endian

print(word.strip(), vec[:5])

the (0.022734079509973526, 0.21401943266391754, -0.05388590320944786, -0.13237027823925018, 0.12502917647361755)


In [3]:
from gensim.models import KeyedVectors

kv = KeyedVectors.load_word2vec_format(path, binary=True)

In [5]:
# Checagem do '@'
print('@' in kv)
print([t for t in kv.index_to_key[:200000] if t.isdigit()][:20])  # amostra de tokens numéricos, se existirem

False
['0', '1', '2', '3', '5', '4', '6', '10', '7', '8', '9', '95', '12', '20', '15', '05', '30', '50', '11', '001']


In [6]:
# Checagem dos tokens estilo Treebank
for tok in ['-LSB-', '-RSB-', '[', ']', '``', "''", '"']:
    print(tok, tok in kv)

-LSB- False
-RSB- False
[ False
] False
`` False
'' False
" False


In [8]:
import sys
sys.path.append('../../src')

from dataset import load_pubmed_rct

X_train, y_train, X_val, y_val, X_test, y_test = load_pubmed_rct()

Repo card metadata block was not found. Setting CardData to empty.


Dentro da função load_pubmed_rct


In [14]:
for s in X_train[:20]:
    print(repr(s))

'To investigate the efficacy of @ weeks of daily low-dose oral prednisolone in improving pain , mobility , and systemic low-grade inflammation in the short term and whether the effect would be sustained at @ weeks in older adults with moderate to severe knee osteoarthritis ( OA ) .'
'A total of @ patients with primary knee OA were randomized @:@ ; @ received @ mg/day of prednisolone and @ received placebo for @ weeks .'
'Outcome measures included pain reduction and improvement in function scores and systemic inflammation markers .'
'Pain was assessed using the visual analog pain scale ( @-@ mm ) .'
'Secondary outcome measures included the Western Ontario and McMaster Universities Osteoarthritis Index scores , patient global assessment ( PGA ) of the severity of knee OA , and @-min walk distance ( @MWD ) .'
'Serum levels of interleukin @ ( IL-@ ) , IL-@ , tumor necrosis factor ( TNF ) - , and high-sensitivity C-reactive protein ( hsCRP ) were measured .'
'There was a clinically relevant

## Tokenização:

In [15]:
#importações da tokenização

import sys
sys.path.append('../../src')

import re
import pickle
from pathlib import Path
from collections import Counter
from itertools import chain

from gensim.models import KeyedVectors
from dataset import load_pubmed_rct

Carregando o embedding pré-treinado:

In [17]:
kv = KeyedVectors.load_word2vec_format(path,
    binary=True
)
print(kv.vector_size, len(kv.index_to_key))  # esperado: 200, 2324849

200 2324849


Carregando o dataset:

In [18]:
X_train, y_train, X_val, y_val, X_test, y_test = load_pubmed_rct()

Repo card metadata block was not found. Setting CardData to empty.


Dentro da função load_pubmed_rct


Funções de pré-processamento, validadas pelo texto real:

In [19]:
padrao_arroba = re.compile(r'@')

def preprocessar(sentenca):
    return [t.lower() for t in sentenca.split()]

def contem_arroba(token):
    return bool(padrao_arroba.search(token))

Tokenizando os splits e mantendo o dado em disco:

In [20]:
def tokenizar_splits(X_train, X_val, X_test, ν):
    return {
        'train': [ν(s) for s in X_train],
        'val': [ν(s) for s in X_val],
        'test': [ν(s) for s in X_test],
    }

tokens_por_split = tokenizar_splits(X_train, X_val, X_test, preprocessar)

out_dir = Path('../../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / 'tokens_pubmed_rct20k.pkl', 'wb') as f:
    pickle.dump(tokens_por_split, f)

Criando vetor sintético para tokens com @:

In [21]:
numeros_no_vocab = [t for t in kv.index_to_key if t.isdigit()]
print(len(numeros_no_vocab))  # tamanho da amostra que sustenta a média

vetor_numerico = sum(kv[t] for t in numeros_no_vocab) / len(numeros_no_vocab)

20824


Medição da cobertura da escolha do @:

In [22]:
todos_tokens = list(chain.from_iterable(
    tokens_por_split['train'] + tokens_por_split['val'] + tokens_por_split['test']
))
contagem = Counter(todos_tokens)
tipos = set(contagem)

tipos_arroba = {t for t in tipos if contem_arroba(t)}
tipos_normais = tipos - tipos_arroba
in_vocab = {t for t in tipos_normais if t in kv}
oov_genuino = tipos_normais - in_vocab

total_tokens = sum(contagem.values())
print(f"Tipos totais: {len(tipos)}")
print(f"Tipos com '@': {len(tipos_arroba)}")
print(f"Tipos normais no vocabulário: {len(in_vocab)}")
print(f"Tipos UNK genuíno: {len(oov_genuino)}")
print(f"Fração de tokens com '@': {sum(contagem[t] for t in tipos_arroba) / total_tokens:.4f}")
print(f"Fração de tokens no vocabulário: {sum(contagem[t] for t in in_vocab) / total_tokens:.4f}")
print(f"Fração de tokens UNK genuíno: {sum(contagem[t] for t in oov_genuino) / total_tokens:.4f}")

Tipos totais: 81145
Tipos com '@': 5323
Tipos normais no vocabulário: 64708
Tipos UNK genuíno: 11114
Fração de tokens com '@': 0.0667
Fração de tokens no vocabulário: 0.7611
Fração de tokens UNK genuíno: 0.1723


Para repetir em qualquer sessão, sem repetir a tokenizaçao:

In [ ]:
with open(Path('../../data/processed/tokens_pubmed_rct20k.pkl'), 'rb') as f:
    tokens_por_split = pickle.load(f)

In [23]:
oov_contagem = Counter({t: contagem[t] for t in oov_genuino})
mais_frequentes = oov_contagem.most_common(50)
for token, freq in mais_frequentes:
    print(f"{freq:>6}  {token!r}")

257246  ','
234985  '.'
171837  ')'
170908  '('
 64666  '%'
 41786  '='
 29210  ';'
 17123  '<'
 12622  ':'
 11125  '-lsb-'
 11107  '-rsb-'
  5128  'vs.'
  4313  '+'
  3502  "'"
  3490  "'s"
  3421  '>'
  2923  '/'
  1416  "''"
  1391  '``'
  1082  'clinicaltrials.gov'
   863  'mg/kg'
   830  '`'
   807  'and/or'
   657  'mg/dl'
   623  '$'
   608  'mg/day'
   602  'ng/ml'
   588  'mmol/l'
   489  'i.e.'
   446  'mg/m'
   374  'mg/d'
   368  'kg/m'
   266  'h.'
   264  'mmol/mol'
   259  'g/kg'
   246  'e.g.'
   232  'pg/ml'
   229  'b.'
   207  'g/dl'
   206  'g/l'
   197  '*'
   190  '&'
   173  '~'
   165  'p.'
   164  'nmol/l'
   161  'g/day'
   159  'g/ml'
   144  'mg/l'
   143  'g/d'
   142  'c.'


In [24]:
for tok in [',', '.', ')', '(', '%', '=', ';', '<', ':', '-', '/', '+', '*', '&', '~', '$', '>', "'", '"']:
    print(tok, tok in kv)

, False
. False
) False
( False
% False
= False
; False
< False
: False
- True
/ False
+ False
* False
& False
~ False
$ False
> False
' False
" False


In [25]:
for tok in ['mg', 'kg', 'dl', 'ml', 'mmol', 'day', 'pg', 'nmol', 'ng']:
    print(tok, tok in kv)

mg True
kg True
dl True
ml True
mmol True
day True
pg True
nmol True
ng True


In [26]:
for tok in ['vs', 'ie', 'eg', 'h', 'b', 'p', 'c']:
    print(tok, tok in kv)

vs True
ie True
eg True
h True
b True
p True
c True


In [27]:
def eh_alfanumerico(token):
    return bool(re.search(r'[a-zA-Z]', token))

oov_texto = {t: f for t, f in oov_contagem.items() if eh_alfanumerico(t)}
oov_texto_ordenado = Counter(oov_texto).most_common(50)
for token, freq in oov_texto_ordenado:
    print(f"{freq:>6}  {token!r}")

frac_pontuacao_pura = 1 - sum(oov_texto.values()) / sum(oov_contagem.values())
print(f"Fração da massa OOV que é pontuação/símbolo puro: {frac_pontuacao_pura:.4f}")

 11125  '-lsb-'
 11107  '-rsb-'
  5128  'vs.'
  3490  "'s"
  1082  'clinicaltrials.gov'
   863  'mg/kg'
   807  'and/or'
   657  'mg/dl'
   608  'mg/day'
   602  'ng/ml'
   588  'mmol/l'
   489  'i.e.'
   446  'mg/m'
   374  'mg/d'
   368  'kg/m'
   266  'h.'
   264  'mmol/mol'
   259  'g/kg'
   246  'e.g.'
   232  'pg/ml'
   229  'b.'
   207  'g/dl'
   206  'g/l'
   165  'p.'
   164  'nmol/l'
   161  'g/day'
   159  'g/ml'
   144  'mg/l'
   143  'g/d'
   142  'c.'
   142  'l.'
   137  'i.v.'
   135  'ml/min'
   131  'ml/kg'
   119  'copies/ml'
   117  'a.'
   114  'u.s.'
   113  'mol/l'
   110  's.'
    93  'd.'
    92  'iu/ml'
    91  'overweight/obese'
    90  'etc.'
    84  'mg/ml'
    79  'v.'
    77  'm/s'
    76  'mg/kg/day'
    72  'm.'
    70  'www.clinicaltrials.gov'
    70  'i.'
Fração da massa OOV que é pontuação/símbolo puro: 0.9402


In [28]:
for tok in ['overweight', 'obese', 'and', 'or']:
    print(tok, tok in kv)

overweight True
obese True
and True
or True


In [29]:
for tok in ['iv', 'us', 'l', 'd', 'v', 'm', 'i', 'etc']:
    print(tok, tok in kv)

iv True
us True
l True
d True
v True
m True
i True
etc True


Tokenização segunda parte:

In [37]:
def separar_barra(token):
    if '/' in token and len(token) > 1:
        return [p for p in token.split('/') if p]
    return [token]

In [38]:
def normalizar_pontos(token, kv):
    if '.' in token:
        sem_ponto = token.replace('.', '')
        if sem_ponto in kv:
            return sem_ponto
    return token

In [39]:
def eh_pontuacao_pura(token):
    if token == '-':
        return False
    return not bool(re.search(r'[a-zA-Z0-9]', token))

In [40]:
def preprocessar_v2(sentenca, kv):
    tokens = [t.lower() for t in sentenca.split()]

    resultado = []
    for t in tokens:
        if contem_arroba(t):
            resultado.append(t)  # vetor sintético aplicado depois, na montagem da matriz
            continue

        if eh_pontuacao_pura(t):
            continue  # descartado: grupo 1, decisão tomada

        partes = separar_barra(t)                          # grupo 2
        partes = [normalizar_pontos(p, kv) for p in partes] # grupo 3
        resultado.extend(partes)

    return resultado

Tokenizando novamente:

In [41]:
tokens_por_split = {
    'train': [preprocessar_v2(s, kv) for s in X_train],
    'val': [preprocessar_v2(s, kv) for s in X_val],
    'test': [preprocessar_v2(s, kv) for s in X_test],
}

with open(out_dir / 'tokens_pubmed_rct20k.pkl', 'wb') as f:
    pickle.dump(tokens_por_split, f)

In [42]:
todos_tokens = list(chain.from_iterable(
    tokens_por_split['train'] + tokens_por_split['val'] + tokens_por_split['test']
))
contagem = Counter(todos_tokens)
tipos = set(contagem)

tipos_arroba = {t for t in tipos if contem_arroba(t)}
tipos_normais = tipos - tipos_arroba
in_vocab = {t for t in tipos_normais if t in kv}
oov_genuino = tipos_normais - in_vocab

total_tokens = sum(contagem.values())
print(f"Tipos totais: {len(tipos)}")
print(f"Tipos com '@': {len(tipos_arroba)}")
print(f"Tipos normais no vocabulário: {len(in_vocab)}")
print(f"Tipos UNK genuíno: {len(oov_genuino)}")
print(f"Fração de tokens com '@': {sum(contagem[t] for t in tipos_arroba) / total_tokens:.4f}")
print(f"Fração de tokens no vocabulário: {sum(contagem[t] for t in in_vocab) / total_tokens:.4f}")
print(f"Fração de tokens UNK genuíno: {sum(contagem[t] for t in oov_genuino) / total_tokens:.4f}")

Tipos totais: 77308
Tipos com '@': 5323
Tipos normais no vocabulário: 65055
Tipos UNK genuíno: 6930
Fração de tokens com '@': 0.0793
Fração de tokens no vocabulário: 0.9139
Fração de tokens UNK genuíno: 0.0068


In [43]:
import numpy as np

comprimentos = [len(s) for s in tokens_por_split['train']]
print(f"Sentenças vazias após descarte: {sum(1 for c in comprimentos if c == 0)}")
print(f"Comprimento médio: {np.mean(comprimentos):.2f}, mediana: {np.median(comprimentos)}, máximo: {max(comprimentos)}")

Sentenças vazias após descarte: 3
Comprimento médio: 22.46, mediana: 20.0, máximo: 220


## Vetorização do vocabulário final

- índice 0 → PAD, vetor de zeros, usado para preencher sequências até um comprimento fixo;
- índice 1 → UNK, para os 6.930 tipos em oov_genuino;
- os 65.055 tipos em in_vocab recebem seus vetores reais do kv;
- os 5.323 tipos que contêm @ recebem todos o mesmo vetor_numerico já calculado, mas via índices distintos no word2idx (tokens diferentes como @ e mg/kg/@ continuam sendo strings diferentes, mesmo compartilhando vetor).

In [44]:
PAD_IDX = 0
UNK_IDX = 1

word2idx = {'<PAD>': PAD_IDX, '<UNK>': UNK_IDX}
ARROBA_IDX = 2
word2idx['<ARROBA>'] = ARROBA_IDX  # todo token com '@' aponta para este único índice

proximo_idx = 3
vetores = [None, None, None]  # placeholders para PAD, UNK, ARROBA; preenchidos abaixo

for t in sorted(in_vocab):  # sorted() só para reprodutibilidade determinística
    word2idx[t] = proximo_idx
    proximo_idx += 1

print(f"Vocabulário final: {len(word2idx)} entradas")

Vocabulário final: 65058 entradas


Inicializar UNK com ruído guassiano pequeno para evitar que todas as palavras desconhecidas estejam na origem do espaço de coordenadas. PAD é um vetor de 0s pois precisa ser indistinguível de ausência de conteúdo. Montando a matriz de embedding:

In [49]:
vocab_size = len(word2idx)
dim = kv.vector_size  # 200

E = np.zeros((vocab_size, dim), dtype=np.float32)

E[UNK_IDX] = np.random.normal(scale=0.1, size=dim).astype(np.float32)
E[ARROBA_IDX] = vetor_numerico

for t, idx in word2idx.items():
    if idx in (PAD_IDX, UNK_IDX, ARROBA_IDX):
        continue
    E[idx] = kv[t]

print(E.shape)  # esperado: (65058, 200)

(65058, 200)


Função de tradução entre sentença tokenizada e sequência de índices:

In [50]:
def sentenca_para_indices(tokens):
    indices = []
    for t in tokens:
        if contem_arroba(t):
            indices.append(ARROBA_IDX)
        elif t in word2idx:
            indices.append(word2idx[t])
        else:
            indices.append(UNK_IDX)
    return indices

X_train_idx = [sentenca_para_indices(s) for s in tokens_por_split['train']]
X_val_idx = [sentenca_para_indices(s) for s in tokens_por_split['val']]
X_test_idx = [sentenca_para_indices(s) for s in tokens_por_split['test']]

Separando a magtriz numérica do word2dix e das sequências de índices:

In [51]:
np.save(out_dir / 'embedding_matrix.npy', E)

with open(out_dir / 'word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

with open(out_dir / 'sequences_pubmed_rct20k.pkl', 'wb') as f:
    pickle.dump({
        'train': X_train_idx, 'val': X_val_idx, 'test': X_test_idx,
        'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    }, f)